
1. Survival Bias Across Socioeconomic Status
How did passenger class (Pclass) influence survival, and does the relationship remain significant after controlling for gender and age?

2. Missing Age Investigation
Are passengers with missing Age values systematically different from those with known ages in terms of class, fare, embarkation port, and survival?

3. Family Size vs Survival
Does traveling alone improve survival chances compared to traveling with small or large families?

4. Fare Outlier Analysis
Are extremely high-ticket fares associated with higher survival rates, or are they merely a reflection of passenger class?

5. Embarkation Port Influence
How does embarkation location (Embarked) impact survival, and is the effect explained by passenger class composition at each port?

6. Cabin Information Value
Does having cabin information recorded correlate with socioeconomic status and survival probability?

7. Children's Advantage Analysis
Did children receive a survival advantage, and at what age threshold does survival probability significantly decrease?

8. Gender-Class Interaction
Which had a stronger effect on survival: gender or socioeconomic status?

9. Ticket Sharing Network Analysis
Do passengers sharing the same ticket number exhibit similar survival outcomes, suggesting family/group travel effects?

10. Build a "Survival Archetype" analysis:
Identify the top 5 passenger profiles with the highest survival rate and the bottom 5 profiles with the lowest survival rate using combinations of:
	• Class
	• Gender
	• Age Group
	• Family Size
	• Embarkation Port

In [6]:
import pandas as pd

df = pd.read_csv("titanic.csv")

print(df.groupby("Pclass")["Survived"].sum())

print(df.groupby(["Pclass","Survived"])["Sex"].value_counts())


Pclass
1    136
2     87
3    119
Name: Survived, dtype: int64
Pclass  Survived  Sex   
1       0         male       77
                  female      3
        1         female     91
                  male       45
2       0         male       91
                  female      6
        1         female     70
                  male       17
3       0         male      300
                  female     72
        1         female     72
                  male       47
Name: count, dtype: int64


In [12]:
# 2. Missing Age Investigation
# Are passengers with missing Age values systematically different from those with known ages in terms of class, fare, embarkation port, and survival?

print(df["Age"].isnull().value_counts())
df["MissingAge"]=df["Age"].isnull()
df["Newfare"]=pd.cut(df["Fare"],bins=[0,150,300])
print(df.groupby(["MissingAge","Pclass","Newfare","Embarked"])["Survived"].value_counts())
 

Age
False    714
True     177
Name: count, dtype: int64
MissingAge  Pclass  Newfare     Embarked  Survived
False       1       (0, 150]    C         1           44
                                          0           19
                                Q         0            1
                                          1            1
                                S         1           55
                                                      ..
True        3       (150, 300]  C         1            0
                                Q         0            0
                                          1            0
                                S         0            0
                                          1            0
Name: count, Length: 72, dtype: int64


C:\Users\A040699\AppData\Local\Temp\ipykernel_3880\2757814660.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(df.groupby(["MissingAge","Pclass","Newfare","Embarked"])["Survived"].value_counts())


In [10]:
# 3. Family Size vs Survival
# Does traveling alone improve survival chances compared to traveling with small or large families?

import pandas as pd

df = pd.read_csv("titanic.csv")
print(df.groupby("Parch")["Survived"].value_counts())
df.groupby("Parch")["Survived"].sum()




df["Alone"] = df["Parch"] == 0


print("Travelling Alone:", df[df["Parch"] == 0]["Survived"].sum())
print("Not Alone:", df[df["Parch"] > 0]["Survived"].sum())

# print("Survival without family:",df.groupby("Parch")["Survived"].sum())


# print("Survival with family:",df.groupby("Parch")["Survived"].sum().drop(0).sum())



Parch  Survived
0      0           445
       1           233
1      1            65
       0            53
2      0            40
       1            40
3      1             3
       0             2
4      0             4
5      0             4
       1             1
6      0             1
Name: count, dtype: int64
Travelling Alone: 233
Not Alone: 109


In [27]:
# 4. Fare Outlier Analysis
# Are extremely high-ticket fares associated with higher survival rates, or are they merely a reflection of passenger class ?

import pandas as pd

df=pd.read_csv("titanic.csv")

Q1=df["Fare"].quantile(0.25)
Q3=df["Fare"].quantile(0.75)
IQR= Q3-Q1

upper = Q3 +1.5*IQR

outliers=df[df["Fare"]>upper]

print("Outliers survived :",outliers["Survived"].sum())
print("Normal Fare survival :",df["Survived"].sum())


# print("Class survival:",outliers["Pclass"],outliers["Survived"].sum())
print("Class survival:",outliers["Pclass"].value_counts())




Outliers survived : 79
Normal Fare survival : 342
Class survival: Pclass
1    104
3      7
2      5
Name: count, dtype: int64


In [4]:
# 5. Embarkation Port Influence
# How does embarkation location (Embarked) impact survival, and is the effect explained by passenger class composition at each port?

import pandas as pd

df=pd.read_csv("titanic.csv")

df["Embarked"].isnull().value_counts()
df=df.dropna(subset=["Embarked"])

print("Survived by port:",df.groupby("Embarked")["Survived"].sum())
print("Survived by port:",df.groupby(["Embarked" , "Pclass"])["Survived"].sum())


Survived by port: Embarked
C     93
Q     30
S    217
Name: Survived, dtype: int64
Survived by port: Embarked  Pclass
C         1         59
          2          9
          3         25
Q         1          1
          2          2
          3         27
S         1         74
          2         76
          3         67
Name: Survived, dtype: int64


In [ ]:
# 6. Cabin Information Value
# Does having cabin information recorded correlate with socioeconomic status and survival probability?

import pandas as pd

df=pd.read_csv("titanic.csv")

df["Cabin"].isnull().value_counts()

print("Cabin with no info,survived:",df[df["Cabin"].isnull()]["Survived"].value_counts())

print("Cabin with info,survived:",df[df["Cabin"].notnull()]["Survived"].value_counts())

print("Cabin Info vs Pclass vs Survival(%):",pd.crosstab(
    [df["Cabin"], df["Pclass"]],
    df["Survived"],
    normalize="index"
) * 100)



Cabin with no info,survived: Survived
0    481
1    206
Name: count, dtype: int64
Cabin with info,survived: Survived
1    136
0     68
Name: count, dtype: int64
Cabin Info vs Pclass vs Survival(%): Survived          0      1
Cabin Pclass              
A10   1       100.0    0.0
A14   1       100.0    0.0
A16   1         0.0  100.0
A19   1       100.0    0.0
A20   1         0.0  100.0
...             ...    ...
F33   2         0.0  100.0
F38   3       100.0    0.0
F4    2         0.0  100.0
G6    3        50.0   50.0
T     1       100.0    0.0

[147 rows x 2 columns]


7. Children's Advantage Analysis
Did children receive a survival advantage, and at what age threshold does survival probability significantly decrease?

In [24]:
import pandas as pd

df=pd.read_csv("titanic.csv")
pd.crosstab(df["Pclass"],["Survived"])


pd.crosstab(df["Pclass"],df["Survived"],normalize="index").round(2)*100

pd.crosstab(df["Pclass"],df["Sex"],df["Survived"],aggfunc="mean")*100


Sex,female,male
Pclass,,
1,96.808511,36.885246
2,92.105263,15.740741
3,50.000000,13.544669


7. Children's Advantage Analysis
Did children receive a survival advantage, and at what age threshold does survival probability significantly decrease?

In [ ]:
import pandas as pd

df=pd.read_csv("titanic.csv")

df["AgeGroups"]=pd.cut(df["Age"],bins=[0,5,15,30,50,80])

print("Survival by Agegroup:",pd.crosstab(df["AgeGroups"],df["Survived"],normalize="index")*100)



Survival by Agegroup: Survived           0          1
AgeGroups                      
(0, 5]     29.545455  70.454545
(5, 15]    53.846154  46.153846
(15, 30]   64.110429  35.889571
(30, 50]   57.676349  42.323651
(50, 80]   65.625000  34.375000


8. Gender-Class Interaction
Which had a stronger effect on survival: gender or socioeconomic status?

In [16]:
import pandas as pd

df=pd.read_csv("titanic.csv")

df["M(0),F(1)"]=df["Sex"].map({"male": 0,"female": 1})

print("Relation of gender with Survival :",(df["M(0),F(1)"].corr(df["Survived"])))

print("Relation of class with Survival :",(df["Pclass"].corr(df["Survived"])))

print("Multiple correlation:",df[["Survived","Pclass","Age","Fare","M(0),F(1)"]].corr()["Survived"])

Relation of gender with Survival : 0.5433513806577546
Relation of class with Survival : -0.3384810359610148
Multiple correlation: Survived     1.000000
Pclass      -0.338481
Age         -0.077221
Fare         0.257307
M(0),F(1)    0.543351
Name: Survived, dtype: float64


9. Ticket Sharing Network Analysis
Do passengers sharing the same ticket number exhibit similar survival outcomes, suggesting family/group travel effects?

In [ ]:
import pandas as pd

df=pd.read_csv("titanic.csv")

df["GroupSize"]=df.groupby("Ticket")["Ticket"].transform("count") #Groups passengers by ticket number.
                                                                #Selects the Ticket column within each group.
                                                                #Counts how many rows are in each ticket group and returns the count for every row in that group.
                                                                #Notice the output has the same number of rows as the original DataFrame.
                                                                # That's why transform() is useful.
df.groupby("GroupSize")["Survived"].value_counts()

# shared=df.groupby("Ticket").filter(lambda x:len(x)>1)

# shared.groupby("Ticket")["Survived"].value_counts()




GroupSize  Survived
1          0           384
           1           163
2          1           108
           0            80
3          1            44
           0            19
4          0            22
           1            22
5          0            10
6          0            18
7          0            16
           1             5
Name: count, dtype: int64

10. Build a "Survival Archetype" analysis:
Identify the top 5 passenger profiles with the highest survival rate and the bottom 5 profiles with the lowest survival rate using combinations of:
	• Class
	• Gender
	• Age Group
	• Family Size
	• Embarkation Port

In [ ]:
import pandas as pd

df=pd.read_csv("titanic.csv")

df["M-F_Numbers"]=df["Sex"].map({"male": 0,"female": 1})

df["AgeGroups"]=pd.cut(df["Age"],bins=[0,5,14,25,45,60,80])

df["Embarked"].isnull().value_counts()
df=df.dropna(subset=["Embarked"])

df["GroupSize"]=df.groupby("Ticket")["Ticket"].transform("count") 
df.groupby("GroupSize")["Survived"].value_counts()

print("Survival by various factors:")
df.groupby(["GroupSize","Pclass","Embarked","AgeGroups","M-F_Numbers"])["Survived"].value_counts()




Survival by various factors:


C:\Users\A040699\AppData\Local\Temp\ipykernel_24668\3063461240.py:16: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(["GroupSize","Pclass","Embarked","AgeGroups","M-F_Numbers"])["Survived"].value_counts()


GroupSize  Pclass  Embarked  AgeGroups  M-F_Numbers  Survived
1          1       C         (0, 5]     0            0           0
                                                     1           0
                                        1            0           0
                                                     1           0
                             (5, 14]    0            0           0
                                                                ..
7          3       S         (45, 60]   1            1           0
                             (60, 80]   0            0           0
                                                     1           0
                                        1            0           0
                                                     1           0
Name: count, Length: 1512, dtype: int64